# CLAP vs SBERT Overlap Analysis

This notebook compares the retrieval results from **CLAP (Audio-Text)** and **SBERT (Metadata-Text)** views.
We use common text queries and measure the Overlap@10 and rank correlation.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Add project root to path
sys.path.append(os.path.abspath(".."))

from src.config import PROCESSED_DIR, FMA_METADATA_DIR
from src.embeddings.clap import CLAPEmbeddingGenerator
from src.embeddings.sbert import SentenceBERTEmbeddingGenerator
from src.indexing.faiss_index import FaissIndex
from src.metadata import load_tracks

### Load Indices and Tracks

In [ ]:
# Load indices
sbert_index = FaissIndex.load(PROCESSED_DIR / "sbert_faiss.index")
clap_index = FaissIndex.load(PROCESSED_DIR / "clap_faiss.index")

# Load generators for text embedding
sbert_gen = SentenceBERTEmbeddingGenerator()
clap_gen = CLAPEmbeddingGenerator()

# Load tracks for metadata display
tracks = load_tracks(FMA_METADATA_DIR)
print("Indices and generators loaded.")

### Compare Retrieval Overlap

In [ ]:
queries = [
    "sad piano ballad",
    "aggressive heavy metal",
    "upbeat electronic dance music",
    "acoustic guitar folk song",
    "chill lo-fi vibes"
]

results_comparison = []

for q in queries:
    # SBERT retrieval
    s_emb = sbert_gen.embed_text([q])
    s_res = sbert_index.query(s_emb, k=10)
    s_tids = [r[0] for r in s_res]
    
    # CLAP retrieval
    c_emb = clap_gen.embed_text([q])
    c_res = clap_index.query(c_emb, k=10)
    c_tids = [r[0] for r in c_res]
    
    # Calculate overlap
    common = set(s_tids) & set(c_tids)
    overlap_p = len(common) / 10 * 100
    
    results_comparison.append({
        "query": q,
        "overlap_p": overlap_p,
        "common_tids": list(common)
    })
    
    print(f"Query: '{q}'")
    print(f"- SBERT Top Result: {tracks.loc[s_tids[0], ('track', 'title')]} by {tracks.loc[s_tids[0], ('artist', 'name')]}")
    print(f"- CLAP Top Result: {tracks.loc[c_tids[0], ('track', 'title')]} by {tracks.loc[c_tids[0], ('artist', 'name')]}")
    print(f"- Overlap@10: {overlap_p:.1f}% \n")

df_comp = pd.DataFrame(results_comparison)
print(f"Average Overlap@10: {df_comp['overlap_p'].mean():.1f}%")